# Tutorial 04: Interactive Visualization Dashboard

This tutorial demonstrates how to use the CarrierCapture interactive dashboard
for real-time exploration of potential energy surfaces and capture coefficients.

**Learning objectives:**
- Launch the Dash-based visualization dashboard
- Explore dashboard features interactively
- Load and fit potentials in real-time
- Calculate capture coefficients with instant feedback
- Export publication-quality figures

**Prerequisites:**
- CarrierCapture installed with visualization dependencies
- Basic familiarity with Tutorials 01-03

In [ ]:
# Imports
import numpy as np

from carriercapture.core.potential import Potential
from carriercapture.core.config_coord import ConfigCoordinate
from carriercapture.visualization.interactive import create_app, run_server
from carriercapture.visualization import (
    plot_potential,
    plot_capture_coefficient,
    plot_configuration_coordinate,
)

print("CarrierCapture visualization modules loaded successfully.")

## 1. Dashboard Overview

The CarrierCapture dashboard provides a web-based interface with several features:

### Main Features:
- **Potential Fitting Tab**: Upload data, fit with various methods, visualize results
- **Capture Calculation Tab**: Set up two potentials, calculate capture coefficients
- **Parameter Scanning Tab**: Run 2D scans over ΔQ and ΔE
- **Comparison Mode**: Compare multiple potentials side-by-side

### Dashboard Controls:
- Real-time parameter sliders
- Dropdown menus for fit types
- Interactive Plotly figures (zoom, pan, hover)
- Export buttons for data and figures

## 2. Launch the Dashboard

There are several ways to launch the dashboard:

### Method 1: From Python (this notebook)

In [ ]:
# Create the Dash app
app = create_app(debug=False)

print("Dashboard app created.")
print("\nTo launch the dashboard, run one of the following:")
print("  1. app.run(port=8050)                    # Blocking, runs in this process")
print("  2. app.run(port=8050, jupyter_mode='tab') # Opens in new browser tab")
print("\nThen open http://127.0.0.1:8050 in your browser.")

In [ ]:
# Uncomment ONE of the following to launch:

# Option A: Run in separate tab (recommended for Jupyter)
# app.run(port=8050, jupyter_mode="tab")

# Option B: Run with inline iframe (may not work in all notebooks)
# app.run(port=8050, jupyter_mode="inline")

# Option C: Run blocking (useful for scripts)
# app.run(port=8050, debug=True)

print("Uncomment one of the lines above to launch the dashboard.")
print("Note: The dashboard runs a server that needs to be stopped manually (Kernel > Interrupt).")

### Method 2: From Command Line (CLI)

The easiest way to launch the dashboard is from the terminal:

```bash
# Basic launch
carriercapture viz

# With custom port
carriercapture viz --port 8080

# With debug mode (auto-reload on code changes)
carriercapture viz --debug

# Open browser automatically
carriercapture viz --open
```

In [ ]:
# You can also launch from notebook using shell command:
# (This runs in background - stop with Kernel > Interrupt)

# !carriercapture viz --port 8050 &

print("To launch via CLI from this notebook, uncomment the line above.")
print("Then navigate to http://127.0.0.1:8050")

## 3. Dashboard Tour

Once the dashboard is running, you'll see several tabs. Here's what each does:

### Tab 1: Potential Fitting

**Features:**
- Upload Q-E data files (CSV, DAT, TXT)
- Or create harmonic/Morse potentials with sliders
- Choose fit type: spline, harmonic, Morse, polynomial
- Adjust fit parameters in real-time
- View potential curve and eigenvalues
- Export fitted potential as JSON

**Workflow:**
1. Upload data or set harmonic parameters
2. Select fit type from dropdown
3. Adjust smoothness/order sliders
4. Click "Solve" to get eigenvalues
5. Export results when satisfied

### Tab 2: Capture Calculation

**Features:**
- Define initial and final state potentials
- Set electron-phonon coupling (W)
- Configure temperature range
- View configuration coordinate diagram
- See capture coefficient vs temperature
- Arrhenius plot with activation energy

**Workflow:**
1. Set up initial state (excited)
2. Set up final state (ground)
3. Adjust W, volume, Q₀ parameters
4. Click "Calculate" to compute C(T)
5. Analyze Arrhenius plot for activation energy

### Tab 3: Parameter Scanning

**Features:**
- Define ΔQ and ΔE ranges
- Run grid-based parameter scan
- View 2D heatmap of capture coefficients
- Identify optimal parameter regions
- Export scan results as NPZ

**Workflow:**
1. Set ΔQ range (min, max, points)
2. Set ΔE range (min, max, points)
3. Configure fixed parameters (ℏω, T, volume)
4. Click "Run Scan"
5. Explore heatmap interactively

### Tab 4: Comparison Mode

**Features:**
- Load multiple potentials
- Overlay plots for comparison
- Compare fit qualities
- Side-by-side eigenvalue spectra

## 4. Programmatic Control

You can prepare data in Python and then visualize it in the dashboard,
or use the static plotting functions for publication figures.

In [ ]:
# Create potentials programmatically
pot_initial = Potential.from_harmonic(
    hw=0.010,    # 10 meV phonon
    Q0=0.0,
    E0=0.6,
    Q_range=(-15, 25),
    npoints=3000,
)

pot_final = Potential.from_harmonic(
    hw=0.010,
    Q0=12.0,
    E0=0.0,
    Q_range=(-15, 25),
    npoints=3000,
)

# Solve
pot_initial.solve(nev=150)
pot_final.solve(nev=60)

print(f"Created and solved potentials:")
print(f"  Initial: {len(pot_initial.eigenvalues)} eigenvalues")
print(f"  Final: {len(pot_final.eigenvalues)} eigenvalues")

In [ ]:
# Use static plotting functions (same as dashboard uses internally)
fig1 = plot_potential(
    pot_initial, 
    show_wavefunctions=True, 
    wf_sampling=5,
    title="Initial State Potential"
)
fig1.show()

In [ ]:
# Configuration coordinate diagram
fig2 = plot_configuration_coordinate(
    pot_initial, 
    pot_final,
    show_crossing=True,
)
fig2.show()

In [ ]:
# Calculate capture and plot
cc = ConfigCoordinate(
    pot_i=pot_initial,
    pot_f=pot_final,
    W=0.068,
    degeneracy=1,
)

temperature = np.linspace(100, 500, 50)
cc.calculate_overlap(Q0=6.0, cutoff=0.25, sigma=0.015)
cc.calculate_capture_coefficient(volume=1e-21, temperature=temperature)

fig3 = plot_capture_coefficient(cc)
fig3.show()

## 5. Exporting Results

### Export Figures

Plotly figures can be exported in various formats:

In [ ]:
# Export to HTML (interactive)
fig3.write_html("capture_coefficient.html")
print("Saved interactive HTML: capture_coefficient.html")

# Export to PNG (static, requires kaleido)
try:
    fig3.write_image("capture_coefficient.png", width=800, height=500, scale=2)
    print("Saved PNG: capture_coefficient.png")
except Exception as e:
    print(f"PNG export requires kaleido: pip install kaleido")

# Export to PDF (requires kaleido)
try:
    fig3.write_image("capture_coefficient.pdf", width=800, height=500)
    print("Saved PDF: capture_coefficient.pdf")
except Exception as e:
    print(f"PDF export requires kaleido: pip install kaleido")

### Export Data

In [ ]:
# Export potential to JSON
pot_initial.to_json("potential_initial.json")
print("Saved potential: potential_initial.json")

# Export capture results to NPZ
np.savez(
    "capture_results.npz",
    temperature=cc.temperature,
    capture_coefficient=cc.capture_coefficient,
    overlap_matrix=cc.overlap_matrix,
)
print("Saved results: capture_results.npz")

## 6. Dashboard Tips & Tricks

### Performance Tips:
- Use fewer grid points (npoints=2000) for faster interactive response
- Reduce nev for quick exploration, increase for final calculations
- Parameter scans with n_jobs=-1 use all CPU cores

### Visualization Tips:
- Double-click on legend items to isolate traces
- Use box select to zoom into regions
- Hover over data points for exact values
- Click camera icon to download current view as PNG

### Troubleshooting:
- If dashboard doesn't load, check port isn't in use: `lsof -i :8050`
- For slow response, try reducing grid resolution
- Clear browser cache if seeing stale visualizations
- Restart kernel if dashboard becomes unresponsive

## 7. Customizing the Dashboard

The dashboard can be customized by modifying themes:

In [ ]:
from carriercapture.visualization.themes import COLORS, get_default_layout

# View available colors
print("Available color scheme:")
for name, color in COLORS.items():
    print(f"  {name}: {color}")

In [ ]:
# Custom styled plot
import plotly.graph_objects as go

fig = go.Figure()

# Add traces with custom colors
fig.add_trace(go.Scatter(
    x=pot_initial.Q,
    y=pot_initial.fit_func(pot_initial.Q),
    mode='lines',
    name='Initial',
    line=dict(color=COLORS['primary'], width=2),
))

fig.add_trace(go.Scatter(
    x=pot_final.Q,
    y=pot_final.fit_func(pot_final.Q),
    mode='lines',
    name='Final',
    line=dict(color=COLORS['secondary'], width=2),
))

# Apply default layout
fig.update_layout(
    **get_default_layout(),
    title="Custom Styled Configuration Coordinate Diagram",
    xaxis_title="Q (amu^0.5·Å)",
    yaxis_title="E (eV)",
)

fig.show()

## Summary

In this tutorial, we learned:

- ✓ How to launch the CarrierCapture dashboard (Python and CLI)
- ✓ Dashboard features: fitting, capture calculation, scanning, comparison
- ✓ Programmatic creation of plots using static functions
- ✓ Exporting figures (HTML, PNG, PDF) and data (JSON, NPZ)
- ✓ Customizing visualizations with themes

**The dashboard is ideal for:**
- Interactive exploration of parameter space
- Quick prototyping before scripted calculations
- Teaching and presentations
- Generating publication-quality figures

**For production calculations:**
- Use the Python API directly (Tutorials 01-03)
- Use the CLI for batch processing
- Script workflows for reproducibility

## Exercises

1. **Launch and explore**: Start the dashboard and try each tab
2. **Upload data**: Create a CSV file with Q,E columns and upload it
3. **Compare fits**: Use comparison mode to overlay spline vs harmonic fits
4. **Run a scan**: Do a 20×20 parameter scan and find the maximum capture coefficient
5. **Export figures**: Generate a publication-ready PNG of your results

In [ ]:
# Exercise 2: Create sample data file for upload
import pandas as pd

# Generate sample data
Q_sample = np.linspace(-10, 20, 30)
E_sample = 0.5 * 0.01 * Q_sample**2 + np.random.normal(0, 0.02, len(Q_sample))

# Save to CSV
df = pd.DataFrame({'Q': Q_sample, 'E': E_sample})
df.to_csv('sample_potential_data.csv', index=False)
print("Created sample_potential_data.csv")
print("Upload this file to the dashboard's Potential Fitting tab.")
print(df.head())

In [ ]:
# Cleanup: Remove generated files (optional)
import os

cleanup_files = [
    'capture_coefficient.html',
    'capture_coefficient.png', 
    'capture_coefficient.pdf',
    'potential_initial.json',
    'capture_results.npz',
    'sample_potential_data.csv',
]

# Uncomment to clean up:
# for f in cleanup_files:
#     if os.path.exists(f):
#         os.remove(f)
#         print(f"Removed {f}")

print("To clean up generated files, uncomment the cleanup code above.")